# 🚀 Text-to-SQL Generator - Colab Deployment

This notebook deploys the Graph-Based RAG Text-to-SQL application with a public URL.

**Steps:**
1. Run all cells in order
2. Get your public URL from the ngrok output
3. Share URL with professor

⏱️ **Setup time**: ~5-10 minutes

⚠️ **Important**: URL expires when you close this notebook or after 8 hours

## Step 1: Install Ollama

In [ ]:
%%bash
# Install zstd (required for Ollama installation)
apt-get update -qq
apt-get install -y -qq zstd

# Install Ollama
curl -fsSL https://ollama.com/install.sh | sh
echo "✅ Ollama installed"

## Step 2: Start Ollama Server (Background)

In [ ]:
import subprocess
import time
import os

# Kill any existing Ollama processes
print("🔄 Stopping any existing Ollama instances...")
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(2)

# Start Ollama server in background with logging
print("Starting Ollama server with logging...")
log_file = open("ollama.log", "w")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Wait for server to start
time.sleep(5)

# Check if it's running
try:
    import requests
    response = requests.get("http://localhost:11434/api/tags", timeout=2)
    print("✅ Ollama server running on http://localhost:11434")
    print("📝 Logs are being saved to ollama.log")
    print("💡 Run Step 2b anytime to view logs")
except:
    print("⚠️  Server might still be starting... check logs in Step 2b")

## Step 2b: Monitor Ollama Logs (Optional)

**Run this cell anytime to check if Ollama is working properly:**

In [ ]:
%%bash

echo "📋 Checking Ollama server status..."
echo ""

# Check if Ollama is running
if pgrep -f "ollama serve" > /dev/null; then
    echo "✅ Ollama server is running"
    echo ""
    
    # Check for log file
    if [ -f "ollama.log" ]; then
        echo "--- Last 30 lines of Ollama logs ---"
        tail -n 30 ollama.log
        echo "------------------------------------"
        echo ""
        echo "💡 To see live logs, run: !tail -f ollama.log"
        echo "   (Press Runtime → Interrupt to stop)"
    else
        echo "⚠️  Log file not found. Server might be logging elsewhere."
    fi
    
    echo ""
    echo "🔍 Testing Ollama API..."
    curl -s http://localhost:11434/api/tags | head -20 || echo "❌ API not responding"
    
else
    echo "❌ Ollama server is NOT running!"
    echo "   Re-run Step 2 to start it"
fi

## Step 3: Upload Your Fine-tuned Model (IMPORTANT!)

**Choose ONE option:**

### Option A: Use your fine-tuned model (RECOMMENDED for best results)
1. Click the **📁 Files** icon in the left sidebar of Colab
2. Click **Upload** button
3. Upload these 2 files from your computer:
   - `phi3-finetuned-q4.gguf` (2GB - your fine-tuned model)
   - `Modelfile` (model configuration)

⏱️ Upload time: ~3-5 minutes

### Option B: Use base phi3 model (faster setup, but less accurate)
Skip the upload - next cells will download phi3 automatically

In [ ]:
%%bash

# Check if fine-tuned model files are uploaded
if [ -f "phi3-finetuned-q4.gguf" ] && [ -f "Modelfile" ]; then
    echo "✅ Fine-tuned model files found!"
    echo "📦 Creating phi3-finetuned model in Ollama..."
    ollama create phi3-finetuned -f Modelfile
    echo "✅ Fine-tuned model ready: phi3-finetuned"
else
    echo "⚠️  Fine-tuned model files not found"
    echo "📥 Pulling base phi3 model instead..."
    ollama pull phi3
    echo "✅ Base model ready: phi3"
    echo ""
    echo "💡 TIP: For better accuracy, upload phi3-finetuned-q4.gguf and Modelfile"
fi

echo ""
echo "📥 Pulling NoSQL translation model..."
ollama pull qwen2.5-coder:3b
echo "✅ NoSQL model ready: qwen2.5-coder:3b"

echo ""
echo "📊 Available models:"
ollama list

## Step 4: Clone Your Repository

In [ ]:
%%bash
# Clone the repository and checkout group18 branch
echo "📥 Cloning repository (branch: group18)..."

# Try cloning (git will create the target directory)
if git clone -b group18 https://github.com/nayanjha16/CodeGen-Implementations.git graphRagTxtToSql; then
    echo "✅ Repository cloned successfully"
elif git clone -b group18 https://github.com/utkarshSinha1910/graphRagTxtToSql.git; then
    echo "✅ Repository cloned successfully"
else
    echo "❌ Clone failed. Repository might be private or branch doesn't exist."
    echo "📝 Solution: Upload your code as ZIP instead"
    exit 1
fi

# Verify ui.py exists in the correct location
cd graphRagTxtToSql
if [ -f "ui.py" ]; then
    echo "✅ ui.py found - app code is ready!"
    echo "📂 Files in repo:"
    ls -1 | head -10
else
    echo "❌ ui.py not found"
    echo "📂 Current directory contents:"
    ls -la
    exit 1
fi

**⚠️ Fix Nested Folders (if needed):**

If you see nested folders, run this cell to flatten the structure:

In [ ]:
%%bash
# Fix nested folder structure if it exists
if [ -d "graphRagTxtToSql/CodeGen-Implementations" ]; then
    echo "🔧 Fixing nested folder structure..."
    
    # Move everything up one level
    mv graphRagTxtToSql/CodeGen-Implementations temp_folder
    rm -rf graphRagTxtToSql
    mv temp_folder graphRagTxtToSql
    
    echo "✅ Fixed! Structure is now flat"
elif [ -d "CodeGen-Implementations" ] && [ ! -d "graphRagTxtToSql" ]; then
    echo "🔧 Renaming folder..."
    mv CodeGen-Implementations graphRagTxtToSql
    echo "✅ Renamed to graphRagTxtToSql"
else
    echo "✅ No nested folders detected"
fi

# Verify final structure
cd graphRagTxtToSql 2>/dev/null || cd CodeGen-Implementations 2>/dev/null || exit 1
echo "📂 Current location: $(pwd)"
echo "📄 Key files:"
ls -1 | grep -E "ui.py|main.py|requirements.txt" || echo "⚠️ Important files not found!"

**⚠️ If Repository is Private:**

If the clone fails, upload your code as ZIP:
1. Download this notebook's parent folder as ZIP on your computer
2. Upload the ZIP to Colab (Files → Upload)
3. Then run: `!unzip -q CodeGen-Implementations.zip && mv CodeGen-Implementations graphRagTxtToSql`

In [ ]:
%%bash
# Alternative: If you uploaded ZIP file
if [ -f "CodeGen-Implementations.zip" ]; then
    echo "📦 Found ZIP file, extracting..."
    unzip -q CodeGen-Implementations.zip
    mv CodeGen-Implementations graphRagTxtToSql 2>/dev/null || true
    echo "✅ Code extracted from ZIP"
fi

# Verify we have the code
if [ -d "graphRagTxtToSql" ]; then
    cd graphRagTxtToSql
    echo "✅ Ready to proceed"
    ls -la | head -10
else
    echo "❌ graphRagTxtToSql folder not found"
    echo "Please either:"
    echo "  1. Make repository public, OR"
    echo "  2. Upload code as ZIP file"
fi

## Step 4b: Download Spider Dataset

**The Spider database is needed for the app to work:**

In [ ]:
%%bash
cd graphRagTxtToSql

# Check if Data/Spider/database exists
if [ ! -d "Data/Spider/database" ]; then
    echo "📥 Downloading Spider dataset (this is required)..."
    
    # Download Spider dataset
    wget -q https://github.com/taoyds/spider/archive/master.zip -O spider.zip
    unzip -q spider.zip
    
    # Move database directory to correct location
    mkdir -p Data/Spider/database
    mv spider-master/database/* Data/Spider/database/ 2>/dev/null || true
    
    # Cleanup
    rm -rf spider.zip spider-master
    
    echo "✅ Spider database downloaded"
else
    echo "✅ Spider database already exists"
fi

# Verify we have some databases
db_count=$(ls Data/Spider/database | wc -l)
echo "📊 Found $db_count databases"

if [ $db_count -lt 10 ]; then
    echo "⚠️  Warning: Database count seems low"
    echo "   Expected: 200+ databases"
else
    echo "✅ Database setup looks good!"
fi

## Step 5: Install Python Dependencies

In [ ]:
%%bash
cd graphRagTxtToSql
pip install -q streamlit networkx requests sqlparse pyngrok
echo "✅ Dependencies installed"

## Step 6: Setup ngrok for Public URL

⚠️ **Optional but Recommended**: Sign up for free ngrok account at https://ngrok.com and paste your auth token below

In [ ]:
from pyngrok import ngrok, conf

# Optional: Set your ngrok auth token for persistent URL
# Get free token from: https://dashboard.ngrok.com/get-started/your-authtoken
# Uncomment and add your token:
# ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")

print("✅ ngrok ready")

## Step 7: Configure Model for Your App

In [ ]:
import os
import subprocess

os.chdir('/content/graphRagTxtToSql')

# Auto-detect which model is available
result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)

if 'phi3-finetuned' in result.stdout:
    model_name = 'phi3-finetuned'
    print("✅ Using fine-tuned model: phi3-finetuned")
elif 'phi3' in result.stdout:
    model_name = 'phi3'
    print("✅ Using base model: phi3")
else:
    model_name = 'phi3'
    print("⚠️  No model found, will use: phi3")

# Set environment variable
os.environ['OLLAMA_MODEL'] = model_name
print(f"✅ Model configured: {model_name}")

## Step 8: Launch Streamlit with Public URL 🚀

**This will give you a public URL to share!**

In [ ]:
import os
import subprocess
import time
import requests
from pyngrok import ngrok

os.chdir('/content/graphRagTxtToSql')

# Start Streamlit in background
print("🚀 Starting Streamlit app...")
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "ui.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for Streamlit to be ready (check multiple times)
print("⏳ Waiting for Streamlit to start...")
max_attempts = 30
for i in range(max_attempts):
    try:
        response = requests.get("http://localhost:8501/_stcore/health", timeout=1)
        if response.status_code == 200:
            print("✅ Streamlit is ready!")
            break
    except:
        pass
    time.sleep(1)
    if i % 5 == 0:
        print(f"   Still waiting... ({i}/{max_attempts}s)")
else:
    # Timeout - but let's check if app is actually responding
    print("⚠️  Health check timeout - checking if app is accessible...")
    try:
        response = requests.get("http://localhost:8501", timeout=2)
        if response.status_code == 200:
            print("✅ App is responding! Proceeding...")
        else:
            print("⚠️  Warning: Streamlit may not be fully ready")
    except:
        print("❌ Streamlit not responding - check for errors below")

# Now create ngrok tunnel
print("🌐 Creating public URL...")
public_url = ngrok.connect(8501, bind_tls=True)

print("\n" + "="*60)
print("🎉 DEPLOYMENT SUCCESSFUL!")
print("="*60)
print(f"\n📱 Public URL: {public_url}")
print(f"\n👉 Share this URL with your professor!")
print("\n⚠️  Keep this notebook running - URL expires when you close it")
print("⏰ Maximum session: ~8 hours\n")
print("="*60)
print("\n💡 Testing the URL now...")
print("   Visit the URL above to verify it works!")
print("   If you see errors, check Streamlit logs in the next cell\n")

# Keep the cell running
streamlit_process.wait()

## Troubleshooting

**If you get errors:**

1. **"Model not found"**: Change model in Step 3 to `phi3:mini` (smaller, faster)
2. **"Connection refused"**: Restart Step 2 (Ollama server)
3. **"ngrok error"**: Add your ngrok auth token in Step 6
4. **"Out of memory"**: Use T4 GPU runtime: Runtime → Change runtime type → T4 GPU

**To stop and restart:**
- Runtime → Interrupt execution
- Then re-run Step 8

## Quick Test: Is Your App Working?

**Run this cell to verify everything is working:**

In [ ]:
import requests
import subprocess

print("🔍 Checking deployment status...\n")

# 1. Check if Streamlit is running
print("1️⃣ Streamlit Status:")
try:
    response = requests.get("http://localhost:8501", timeout=2)
    print(f"   ✅ Streamlit responding (status: {response.status_code})")
except Exception as e:
    print(f"   ❌ Streamlit not responding: {e}")

# 2. Check if Ollama is running
print("\n2️⃣ Ollama Status:")
try:
    response = requests.get("http://localhost:11434/api/tags", timeout=2)
    print(f"   ✅ Ollama responding")
    # Show available models
    result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
    print(f"   Models: {result.stdout.strip()}")
except Exception as e:
    print(f"   ❌ Ollama not responding: {e}")

# 3. Check ngrok tunnel
print("\n3️⃣ ngrok Tunnel:")
try:
    response = requests.get("http://localhost:4040/api/tunnels", timeout=2)
    data = response.json()
    if data.get('tunnels'):
        tunnel = data['tunnels'][0]
        print(f"   ✅ Public URL: {tunnel['public_url']}")
    else:
        print("   ⚠️  No active tunnels")
except Exception as e:
    print(f"   ⚠️  ngrok API not available: {e}")

# 4. Test the public URL
print("\n4️⃣ Public URL Test:")
print("   👉 Visit your ngrok URL in a browser to verify!")
print("   Expected: You should see the Text-to-SQL interface")

print("\n" + "="*60)
print("✅ If all checks passed, your app is ready!")
print("❌ If any failed, see troubleshooting below")
print("="*60)

## Debug: Check Streamlit Errors

**If Streamlit isn't working, run this to see error logs:**

In [ ]:
%%bash
cd /content/graphRagTxtToSql

echo "🔍 Checking if Streamlit can run..."
echo ""

# Try to run streamlit and capture errors
timeout 10 streamlit run ui.py --server.port 8502 --server.headless true 2>&1 | head -50 || true

echo ""
echo "======================================"
echo "Common Issues:"
echo "1. Missing Data/Spider directory"
echo "2. Import errors (missing packages)"
echo "3. Syntax errors in ui.py"
echo "======================================"

## Optional: Check Server Status

In [ ]:
%%bash
# Check if Ollama is running
curl -s http://localhost:11434/api/tags | head -20

# Check ngrok tunnels
curl -s http://localhost:4040/api/tunnels